# Interactive Experiment Results Browser

This notebook provides an interactive interface to browse, filter, and visualize results from Political Compass experiments. It scans for experiment configurations in the current directory and its subdirectories, allows filtering based on parameters, and generates combined plots for selected experiments.

In [ ]:
import collections
import glob
import json
import os
from multiprocessing import Pool
from typing import Any, Dict, List, Tuple

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ipywidgets import interactive

## Helper Functions

These functions handle finding and loading configuration files, reading experiment scores, and plotting the results.

In [ ]:
def find_config_files(results_dir: str) -> List[str]:
    """
    Scans the results directory and returns a list of paths to all config.json files.
    """
    config_paths = []
    for root, dirs, files in os.walk(results_dir):
        if 'config.json' in files:
            config_paths.append(os.path.join(root, 'config.json'))
    return config_paths


def load_config(file_path: str) -> Dict[str, Any]:
    """
    Loads a single config.json file and extracts the experiment ID.
    """
    experiment_id = os.path.basename(os.path.dirname(file_path))
    try:
        with open(file_path, 'r') as f:
            config_data = json.load(f)
        config_data['experiment_id'] = experiment_id
        return config_data
    except json.JSONDecodeError:
        print(f"Warning: Could not decode JSON from {file_path}. It might be empty.")
        return {'experiment_id': experiment_id} # Return ID to allow filtering
    except Exception as e:
        print(f"An error occurred while processing {file_path}: {e}")
        return None


def load_all_configs(results_dir: str) -> pd.DataFrame:
    """
    Finds all config.json files, loads them in parallel, and returns a DataFrame.
    """
    config_files = find_config_files(results_dir)
    if not config_files:
        print("No 'config.json' files found in the specified directory.")
        return pd.DataFrame()

    # Use multiprocessing to load files in parallel
    with Pool() as pool:
        all_configs = pool.map(load_config, config_files)

    # Filter out any configs that failed to load and create a DataFrame
    valid_configs = [config for config in all_configs if config is not None]
    if not valid_configs:
        return pd.DataFrame()
    return pd.DataFrame(valid_configs)


import pandas as pd
import ipywidgets as widgets
from ipywidgets import interactive, VBox, HBox
from IPython.display import display, clear_output
import collections
import os
import glob
import json
import matplotlib.pyplot as plt
from typing import List, Tuple, Dict, Any

# Make sure to run this magic command in a separate cell in your notebook
# %matplotlib widget

# --- All your helper functions (find_config_files, load_config, etc.) go here ---

def read_scores(file_path: str) -> Tuple[float, float]:
    """
    Reads economic and social scores from a text file.
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()
    econ_score = float(lines[0].strip().split()[1])
    social_score = float(lines[1].strip().split()[1])
    return econ_score, social_score

def plot_combined_results(results: List[Tuple[float, float, str, Dict[str, Any]]], title: str) -> None:
    """
    Generates an interactive plot with static labels and detailed hover annotations.
    """
    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.axhline(0, color='grey', lw=1)
    ax.axvline(0, color='grey', lw=1)

    # Quadrant backgrounds and labels
    ax.fill_between([-10, 0], 0, 10, color='red', alpha=0.2)
    ax.text(-9.5, 9.5, 'Authoritarian Left', ha='left', va='top', fontsize=12)
    ax.fill_between([0, 10], 0, 10, color='blue', alpha=0.2)
    ax.text(9.5, 9.5, 'Authoritarian Right', ha='right', va='top', fontsize=12)
    ax.fill_between([-10, 0], -10, 0, color='green', alpha=0.2)
    ax.text(-9.5, -9.5, 'Libertarian Left', ha='left', va='bottom', fontsize=12)
    ax.fill_between([0, 10], -10, 0, color='purple', alpha=0.2)
    ax.text(9.5, -9.5, 'Libertarian Right', ha='right', va='bottom', fontsize=12)

    ax.set_xlabel("Economic (Left <-> Right)")
    ax.set_ylabel("Social (Libertarian <-> Authoritarian)")
    ax.set_title(title)

    # --- Interactivity and Static Labels ---

    plotted_points = []
    # The 'results' tuple now contains: (econ_score, social_score, description, full_config)
    for econ_score, social_score, description, config in results:
        # 1. Add the static text label (model + persona)
        ax.text(econ_score, social_score + 0.3, description, ha='center', fontsize=8)
        
        # 2. Plot the point and attach the full config for hovering
        point = ax.plot(econ_score, social_score, 'o', markersize=8, picker=5)[0]
        # Format the full config dictionary into a readable string for the annotation
        config_str = json.dumps(config, indent=2)
        point.full_config = config_str # Attach formatted config string to the artist
        plotted_points.append(point)

    # Create a reusable annotation object
    annot = ax.annotate("", xy=(0,0), xytext=(20,20), textcoords="offset points",
                        bbox=dict(boxstyle="round", fc="lightyellow", ec="black", lw=1),
                        arrowprops=dict(arrowstyle="->"))
    annot.set_visible(False)

    def update_annot(point):
        """Updates the annotation's position and text with the full config."""
        annot.xy = point.get_data()
        annot.set_text(point.full_config)
        annot.get_bbox_patch().set_alpha(0.9)

    def on_hover(event):
        """Event handler for mouse motion to show/hide the annotation."""
        vis = annot.get_visible()
        if event.inaxes == ax:
            for point in plotted_points:
                cont, _ = point.contains(event)
                if cont:
                    update_annot(point)
                    annot.set_visible(True)
                    fig.canvas.draw_idle()
                    return
        
        if vis:
            annot.set_visible(False)
            fig.canvas.draw_idle()

    fig.canvas.mpl_connect("motion_notify_event", on_hover)

    plt.grid(True)
    plt.tight_layout()
    plt.show()


## Interactive UI

This function creates and wires up all the `ipywidgets` for the user interface.

In [ ]:
%matplotlib widget

In [ ]:

def create_ui(df_configs: pd.DataFrame):
    """
    Creates the full interactive UI for filtering, selecting, and plotting.
    (This function is mostly the same, with a key change in on_plot_button_clicked)
    """
    if df_configs.empty:
        print("DataFrame is empty, cannot create interactive widgets.")
        return

    style = {'description_width': 'initial'}
    shared_state = {'filtered_df': df_configs.copy(), 'checkboxes': [], 'persistent_selection': set()}
    
    # --- All widget definitions from before ---
    filter_widgets_dict = collections.OrderedDict()
    filter_widgets_dict['experiment_id_contains'] = widgets.Text(value='', placeholder='e.g., 2025_10_16', description='Experiment ID Contains:', style=style)
    potential_filter_keys = [col for col in df_configs.columns if df_configs[col].dtype == 'object' and col not in ['experiment_id', 'output_file_name', 'timestamp']]
    for key in potential_filter_keys:
        options = ['All'] + sorted([str(v) for v in df_configs[key].unique()])
        filter_widgets_dict[key] = widgets.Dropdown(options=options, value='All', description=key, style=style)
    select_all_checkbox = widgets.Checkbox(value=False, description='Select/Deselect All Visible', indent=False)
    add_to_selection_button = widgets.Button(description="Add to Master Selection", icon='plus-square', button_style='info')
    clear_selection_button = widgets.Button(description="Clear Master Selection", icon='trash', button_style='danger')
    selection_management_box = HBox([select_all_checkbox, add_to_selection_button, clear_selection_button])
    scoring_method_widget = widgets.Dropdown(options=['max_prob', 'weighted'], value='max_prob', description='Scoring Method:', style=style)
    plot_button = widgets.Button(description="Plot Master Selection", icon='line-chart', button_style='success')
    output_area = widgets.Output()
    selection_output_area = widgets.Output()
    plot_output_area = widgets.Output()

    # --- All event handlers from before (on_select_all, on_add_to_selection, etc.) ---
    def update_selection_output():
        with selection_output_area:
            selection_output_area.clear_output(wait=True)
            count = len(shared_state['persistent_selection'])
            print(f"✅ Master selection contains {count} experiment(s).")
    def on_select_all_toggled(change):
        for cb in shared_state['checkboxes']: cb.value = change.new
    def on_add_to_selection_clicked(b):
        for cb in shared_state['checkboxes']:
            if cb.value: shared_state['persistent_selection'].add(cb.description)
        update_selection_output()
    def on_clear_selection_clicked(b):
        shared_state['persistent_selection'].clear()
        for cb in shared_state['checkboxes']: cb.value = False
        update_selection_output()
    select_all_checkbox.observe(on_select_all_toggled, names='value')
    add_to_selection_button.on_click(on_add_to_selection_clicked)
    clear_selection_button.on_click(on_clear_selection_clicked)
    def filter_and_display(**filters):
        # This function remains the same as before
        filtered_df = df_configs.copy()
        id_contains_value = filters.get('experiment_id_contains', '').strip()
        if id_contains_value:
            filtered_df = filtered_df[filtered_df['experiment_id'].str.contains(id_contains_value, na=False)]
        for key, value in filters.items():
            if key != 'experiment_id_contains' and value != 'All':
                filtered_df = filtered_df[filtered_df[key].astype(str) == value]
        shared_state['filtered_df'] = filtered_df
        with output_area:
            output_area.clear_output(wait=True)
            print(f"Found {len(filtered_df)} matching experiments.")
            checkboxes = [widgets.Checkbox(value=(exp_id in shared_state['persistent_selection']), description=exp_id, indent=False) for exp_id in filtered_df['experiment_id']]
            shared_state['checkboxes'] = checkboxes
            checkbox_container = VBox(children=checkboxes)
            display(checkbox_container)
            select_all_checkbox.value = False

    # --- MODIFIED on_plot_button_clicked ---
    def on_plot_button_clicked(b):
        with plot_output_area:
            plot_output_area.clear_output(wait=True)
            selected_ids = sorted(list(shared_state['persistent_selection']))
            if not selected_ids:
                print("Master selection is empty. Add experiments before plotting.")
                return
            print(f"Plotting {len(selected_ids)} experiments from master selection...")
            scoring_method = scoring_method_widget.value
            results_data = []
            for exp_id in selected_ids:
                search_pattern = os.path.join('.', '*', exp_id, f"results_{scoring_method}.txt")
                found_files = glob.glob(search_pattern)
                if not found_files:
                    print(f"Warning: Could not find results file for {exp_id}")
                    continue
                try:
                    econ_score, social_score = read_scores(found_files[0])
                    config_row = df_configs[df_configs['experiment_id'] == exp_id].iloc[0]
                    description = f"{config_row.get('model_name', 'N/A')}\n{config_row.get('persona', 'N/A')}"
                    
                    # *** KEY CHANGE HERE ***
                    # We now pass the full config dictionary along with the other data.
                    # The config_row is a pandas Series, so we convert it to a dict.
                    full_config = config_row.to_dict()
                    results_data.append((econ_score, social_score, description, full_config))

                except Exception as e:
                    print(f"Error processing file for {exp_id}: {e}")
            if results_data:
                plot_title = f"Combined Political Compass Results ({scoring_method})"
                plot_combined_results(results_data, plot_title)
            else:
                print("No valid data could be collected for the selected experiments.")

    plot_button.on_click(on_plot_button_clicked)
    interactive_filter = interactive(filter_and_display, **filter_widgets_dict)
    update_selection_output()
    
    # Display the full UI (same as before)
    display(
        interactive_filter,
        selection_management_box,
        output_area,
        HBox([scoring_method_widget, plot_button]),
        selection_output_area,
        plot_output_area
    )

## Main Execution

This cell loads the initial data and kicks off the UI. Run this to start the application.

In [ ]:
# --- Load the data ---
print("Scanning for experiment configurations...")
# Assuming the notebook is run from the 'results' directory
df_all_configs = load_all_configs('.')

In [ ]:
if not df_all_configs.empty:
    print(f"Successfully loaded {len(df_all_configs)} experiment configurations.")
    create_ui(df_all_configs)
else:
    print("Could not load any experiment configurations.")